# T/NK Cell Subcluster Annotation + scANVI Retrain
Version: v1.1 (2026-03-18)

**Changes vs v1.0**:
  [P0-1] NEW_LABELS_KEY initialization: inherit OLD_LABELS_KEY first, then
         only clear cells in TARGET_CELL_TYPES before re-filling from annotation map.
         Prevents non-target L2 cells from silently becoming Unknown.
  [P0-2] Latent consistency check now compares reloaded scVI against X_scvi
         (not X_scanvi). X_scvi vs X_scanvi comparison is meaningless (different spaces).
  [P0-3] Added explicit guard: raise before scANVI train if labeled_only has
         0 or 1 classes. Prevents silent misuse of supervision-free / single-class training.
  [P1-1] SUBCLUSTER_REP default changed to X_scvi. Using X_scanvi for de novo
         subcluster discovery introduces circularity with old supervision.
  [P1-2] TARGET_CELL_TYPES now defaults to explicit whitelist matching the
         prepared annotation maps, not None (avoids silent Unknown inflation).
  [P1-3] ANNOTATION_RES_OVERRIDE applied BEFORE printing annotation templates.
         Users always see cluster IDs from the final resolution they will annotate.
  [P1-4] Output header corrected: scVI is NOT retrained; only scANVI is saved.
         Original SCVI_MODEL_DIR is explicitly documented as the scVI reference.
  [P2-1] n_assigned now only counts non-Unknown mapped cells.
  [P2-2] Removed dead variable rep_matrix.
  [P2-3] n_samples_per_label clamped to min(100, min_class_size) not fixed floor 10.
  [P2-4] Thread env vars and sc.settings.n_jobs reduced to avoid oversubscription.

**Input**:
  - adata_tnk_scanvi_ref_20260315_v1_2.h5ad
    (post-scVI/scANVI, HVG subset + .raw full gene log1p)
  - tnk_scvi_ref_model/  (original trained scVI; not retrained, reused as-is)

**Output**:
  - subcluster_markers/<L2_type>/markers_leiden_rX.X.csv
  - subcluster_markers/<L2_type>/dotplot_datadriven_r*.pdf
  - subcluster_markers/<L2_type>/dotplot_known_markers_r*.pdf
  - subcluster_markers/<L2_type>/matrixplot_known_markers_r*.pdf
  - subcluster_markers/<L2_type>/featureplot_known_markers.pdf
  - subcluster_markers/<L2_type>/dotplot_data_r*_{mean_expr_raw,mean_expr_scaled,pct_expressing}.csv
  - adata_tnk_scanvi_ref_retrain_v1_1.h5ad
  - tnk_scanvi_ref_model_retrain/   (new scANVI with refined labels)
  NOTE: scVI is NOT retrained. Use original SCVI_MODEL_DIR for future scArches queries.

**QRM rules applied**:
  - basis='umap' (NOT 'X_umap') in sc.pl.embedding
  - sc.settings.vector_friendly=True before plot blocks; no rasterized= kwarg
  - use_raw=True for rank_genes_groups (full gene .raw)
  - QRM 15.1 : reference uses from_scvi_model (correct for reference path)
  - QRM 13   : category dtype before write_h5ad

## 0. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION  (edit here only)
# ============================================================================

INPUT_H5AD     = "/home/h2048/data/py/0315/tnk_scarches_ref/adata_tnk_scanvi_ref_20260315_v1_2.h5ad"
SCVI_MODEL_DIR = "/home/h2048/data/py/0315/tnk_scarches_ref/tnk_scvi_ref_model"
OUTPUT_DIR     = "/home/h2048/data/py/0318/tnk_subcluster_retrain"

# Column used to split the dataset into per-lineage subsets for subclustering
SPLIT_KEY         = "scanvi_label"          # 按 L3 标签分组
TARGET_CELL_TYPES = None                    # None = 自动取所有 L3 标签

# [P1-1 fix] Use X_scvi (batch-corrected VAE latent) for de novo subcluster discovery.
# Rationale: X_scanvi is shaped by old supervision labels and introduces circularity
# when used to discover new subpopulations. X_scvi is cleaner for this task.
# Switch to X_scanvi only if X_scvi is absent or shows poor batch mixing.
SUBCLUSTER_REP = "X_scanvi"

# Leiden resolutions to test per cell type
LEIDEN_RESOLUTIONS = [1.0]

# Default resolution for annotation; override per cell type via ANNOTATION_RES_OVERRIDE
DEFAULT_ANNOTATION_RESOLUTION = 1.0

# Batch key (used for scVI reload setup_anndata)
BATCH_KEY      = "sample"

# Existing scANVI labels column in input h5ad (inherited for non-target cell types)
OLD_LABELS_KEY = "scanvi_label"

# New refined scANVI labels column (written to output h5ad)
NEW_LABELS_KEY = "scanvi_label_refined"
UNLABELED      = "Unknown"

# scANVI retrain parameters
SCANVI_EPOCHS  = 200
BATCH_SIZE     = 256
N_SAMPLES_PER_LABEL = None   # None = auto (min(100, min_class_size * 0.8))

# Marker detection
MARKER_METHOD         = "wilcoxon"
N_TOP_MARKERS         = 20
MIN_IN_GROUP_FRACTION = 0.1

# Visualization
DPI  = 300
SEED = 42

print("Configuration loaded")
print(f"  Input h5ad        : {INPUT_H5AD}")
print(f"  scVI model        : {SCVI_MODEL_DIR}")
print(f"  Output            : {OUTPUT_DIR}")
print(f"  Target cell types : {TARGET_CELL_TYPES}")
print(f"  Subcluster rep    : {SUBCLUSTER_REP}")

Configuration loaded
  Input h5ad        : /home/h2048/data/py/0315/tnk_scarches_ref/adata_tnk_scanvi_ref_20260315_v1_2.h5ad
  scVI model        : /home/h2048/data/py/0315/tnk_scarches_ref/tnk_scvi_ref_model
  Output            : /home/h2048/data/py/0318/tnk_subcluster_retrain
  Target cell types : ['NK cells', 'CD4 T cells', 'CD8 T cells']
  Subcluster rep    : X_scanvi


## 1. Imports & Setup

In [2]:
import os, gc, warnings
warnings.filterwarnings("ignore")

# [P2-4 fix] Reduced thread counts to avoid oversubscription on shared HPC nodes
os.environ["OMP_NUM_THREADS"]      = "4"
os.environ["MKL_NUM_THREADS"]      = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import scipy.sparse as sparse
import scanpy as sc
import scvi
from pathlib import Path

sc.settings.verbosity = 2
sc.settings.n_jobs    = 8   # [P2-4 fix] match OMP threads, avoid CPU oversubscription
sc.settings.set_figure_params(dpi=DPI, facecolor="white", frameon=False)

np.random.seed(SEED)
scvi.settings.seed = SEED

output_dir = Path(OUTPUT_DIR)
marker_dir = output_dir / "subcluster_markers"
fig_dir    = output_dir / "figures"
for d in [output_dir, marker_dir, fig_dir]:
    d.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(fig_dir)

print(f"scanpy : {sc.__version__}")
print(f"scvi   : {scvi.__version__}")
print(f"Output : {output_dir}")

Seed set to 42


scanpy : 1.11.5
scvi   : 1.3.3
Output : /home/h2048/data/py/0318/tnk_subcluster_retrain


## 2. Load Data

In [3]:
print("Loading h5ad...")
adata = sc.read_h5ad(INPUT_H5AD)
print(f"  Shape  : {adata.shape}")
print(f"  obsm   : {list(adata.obsm.keys())}")
print(f"  layers : {list(adata.layers.keys()) if adata.layers else 'None'}")
if adata.raw is not None:
    print(f"  .raw   : {adata.raw.n_obs} x {adata.raw.n_vars} genes")
else:
    print("  .raw   : None")

# Verify required keys exist before proceeding
assert SUBCLUSTER_REP in adata.obsm, \
    f"'{SUBCLUSTER_REP}' not in obsm. Available: {list(adata.obsm.keys())}"
assert SPLIT_KEY in adata.obs.columns, \
    f"'{SPLIT_KEY}' not in obs.columns"

# Verify all TARGET_CELL_TYPES exist in SPLIT_KEY
missing_types = [ct for ct in TARGET_CELL_TYPES
                 if ct not in adata.obs[SPLIT_KEY].values]
if missing_types:
    raise ValueError(
        f"TARGET_CELL_TYPES not found in obs['{SPLIT_KEY}']: {missing_types}\n"
        f"Available: {sorted(adata.obs[SPLIT_KEY].unique().tolist())}"
    )

print(f"\nTarget cell types ({SPLIT_KEY}):")
for ct in TARGET_CELL_TYPES:
    n = (adata.obs[SPLIT_KEY] == ct).sum()
    print(f"  {ct}: {n:,} cells")

Loading h5ad...


KeyboardInterrupt: 

## 3. Per-Cell-Type Subclustering on X_scvi Space

For each L2 cell type in TARGET_CELL_TYPES:
  1. Extract subset; compute within-type neighbors on SUBCLUSTER_REP (X_scvi)
  2. Run Leiden at multiple resolutions
  3. Compute rank_genes_groups markers (use_raw=True)
  4. Save marker CSVs, UMAP overview, known-marker dotplots + featureplots
  5. Store chosen-resolution cluster IDs back into main adata.obs

In [ ]:
def sanitize_colname(s):
    return s.lower().replace(" ", "_").replace("/", "_").replace("+", "plus")

subcluster_summary  = {}   # {cell_type: {annotation_key, annotation_res, n_subclusters, ...}}
res_results_global  = {}   # {cell_type: {resolution: n_clusters}} -- used by CSV export cell

for cell_type in TARGET_CELL_TYPES:
    print(f"\n{'='*60}")
    print(f"Subclustering: {cell_type}")
    print(f"{'='*60}")

    mask    = adata.obs[SPLIT_KEY] == cell_type
    adata_s = adata[mask].copy()
    n_cells = adata_s.n_obs
    print(f"  Cells: {n_cells:,}")

    if n_cells < 50:
        print(f"  [SKIP] < 50 cells -- skipping {cell_type}")
        continue

    ct_safe = sanitize_colname(cell_type)
    ct_mdir = marker_dir / ct_safe
    ct_mdir.mkdir(exist_ok=True)

    # [P2-2 fix] Removed dead variable rep_matrix

    # Within-subset neighbors on SUBCLUSTER_REP
    n_neighbors = min(30, max(5, n_cells // 5))
    print(f"  Neighbors on {SUBCLUSTER_REP} (n_neighbors={n_neighbors})...")
    sc.pp.neighbors(adata_s, use_rep=SUBCLUSTER_REP, n_neighbors=n_neighbors)

    # Within-subset UMAP for visualization
    sc.tl.umap(adata_s, min_dist=0.3, spread=1.0)

    # Multi-resolution Leiden
    res_results = {}
    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_r{res}"
        try:
            sc.tl.leiden(adata_s, resolution=res, key_added=key)
            n_clust = adata_s.obs[key].nunique()
            res_results[res] = n_clust
            print(f"  Leiden r={res}: {n_clust} clusters")
        except Exception as e:
            print(f"  Leiden r={res} failed: {e}")

    if not res_results:
        print(f"  [ERROR] All Leiden runs failed for {cell_type} -- skipping")
        del adata_s; gc.collect()
        continue

    # Marker genes for each resolution
    for res in LEIDEN_RESOLUTIONS:
        key    = f"leiden_r{res}"
        n_clust = res_results.get(res, 0)
        if key not in adata_s.obs.columns or n_clust <= 1:
            continue

        print(f"  Computing markers r={res} ({n_clust} clusters)...")
        try:
            sc.tl.rank_genes_groups(
                adata_s,
                groupby   = key,
                method    = MARKER_METHOD,
                use_raw   = True,
                pts       = True,
                key_added = f"rgg_{key}",
            )
            all_markers = []
            for clust in adata_s.obs[key].cat.categories:
                df = sc.get.rank_genes_groups_df(
                    adata_s, group=clust, key=f"rgg_{key}",
                    pval_cutoff=0.05, log2fc_min=0.25,
                )
                if df.empty:
                    continue
                if "pct_nz_group" in df.columns:
                    df = df[df["pct_nz_group"] >= MIN_IN_GROUP_FRACTION]
                df.insert(0, "cluster",    clust)
                df.insert(1, "cell_type",  cell_type)
                df.insert(2, "resolution", res)
                all_markers.append(df.head(N_TOP_MARKERS))

            if all_markers:
                markers_df = pd.concat(all_markers, ignore_index=True)
                csv_path   = ct_mdir / f"markers_leiden_r{res}.csv"
                markers_df.to_csv(csv_path, index=False)
                print(f"    Saved: {csv_path.name}")
            else:
                print(f"    No significant markers for r={res}")

        except Exception as e:
            print(f"    Markers r={res} failed: {e}")

    # UMAP overview: reference labels + each resolution
    sc.settings.vector_friendly = True
    n_res_valid = sum(1 for r in LEIDEN_RESOLUTIONS
                      if f"leiden_r{r}" in adata_s.obs.columns)
    ncols   = min(n_res_valid + 2, 4)
    nrows   = int(np.ceil((n_res_valid + 2) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4.5))
    ax_flat = np.array(axes).ravel()
    ax_idx  = 0

    if "cell_type_L3" in adata_s.obs.columns:
        sc.pl.embedding(adata_s, basis="umap", color="cell_type_L3",
                        title="L3 (reference labels)",
                        ax=ax_flat[ax_idx], show=False,
                        legend_loc="right margin", legend_fontsize=6)
        ax_idx += 1

    if OLD_LABELS_KEY in adata_s.obs.columns:
        sc.pl.embedding(adata_s, basis="umap", color=OLD_LABELS_KEY,
                        title=f"Old scANVI ({OLD_LABELS_KEY})",
                        ax=ax_flat[ax_idx], show=False,
                        legend_loc="right margin", legend_fontsize=6)
        ax_idx += 1

    for res in LEIDEN_RESOLUTIONS:
        key = f"leiden_r{res}"
        if key not in adata_s.obs.columns:
            continue
        sc.pl.embedding(adata_s, basis="umap", color=key,
                        title=f"Leiden r={res} ({res_results.get(res,'?')} clusters)",
                        ax=ax_flat[ax_idx], show=False,
                        legend_loc="on data", legend_fontsize=7)
        ax_idx += 1

    for i in range(ax_idx, len(ax_flat)):
        ax_flat[i].set_visible(False)

    plt.suptitle(f"{cell_type} -- Subclustering Overview", fontsize=13, y=1.01)
    plt.tight_layout()
    fig.savefig(ct_mdir / "umap_resolutions_overview.pdf",
                bbox_inches="tight", dpi=DPI)
    plt.close(fig)
    print(f"  Saved: umap_resolutions_overview.pdf")

    # Dotplot 1: data-driven top markers at DEFAULT resolution
    chosen_key = f"leiden_r{DEFAULT_ANNOTATION_RESOLUTION}"
    if chosen_key in adata_s.obs.columns:
        rgg_key = f"rgg_{chosen_key}"
        if rgg_key in adata_s.uns:
            try:
                sc.settings.vector_friendly = True
                dp = sc.pl.rank_genes_groups_dotplot(
                    adata_s, n_genes=5, key=rgg_key,
                    groupby=chosen_key, use_raw=True,
                    show=False, return_fig=True,
                )
                dp.savefig(
                    ct_mdir / f"dotplot_datadriven_r{DEFAULT_ANNOTATION_RESOLUTION}.pdf",
                    bbox_inches="tight", dpi=DPI)
                plt.close()
                print(f"  Saved: dotplot_datadriven_r{DEFAULT_ANNOTATION_RESOLUTION}.pdf")
            except Exception as e:
                print(f"  Data-driven dotplot failed: {e}")

    # Dotplot 2: curated known T/NK markers for all resolutions
    KNOWN_MARKERS_ORDERED = [
        # Lineage
        "CD3D", "CD3E",
        "CD4",
        "CD8A", "CD8B",
        "GNLY", "NKG7", "NCAM1", "FCGR3A",
        "TRDC", "TRGC1",
        "KLRB1", "ZBTB16",
        "KIT", "IL7R",
        # Naive / central memory
        "CCR7", "SELL", "TCF7", "LEF1", "MAL", "IL6R",
        # Effector / TEM
        "GZMK", "GZMA", "GZMH", "GZMB", "PRF1",
        "IFNG", "TNF",
        # TRM
        "ITGAE", "ITGA1", "CD69", "CXCR6", "RGS1", "ZNF683",
        # TEMRA / terminal
        "KLRG1", "CX3CR1", "S1PR5", "ZEB2", "FGFBP2",
        # Exhaustion / inhibitory
        "PDCD1", "HAVCR2", "TIGIT", "LAG3", "TOX", "ENTPD1",
        # Tfh / Treg
        "CXCR5", "CXCL13", "TOX2",
        "FOXP3", "IL2RA", "CTLA4",
        # Th1 / Th17
        "TBX21", "RORC", "IL23R", "CCR6",
        # NK functional
        "XCL1", "XCL2", "AREG",
        "ISG15", "IFIT1", "MX1",
        # Proliferation
        "MKI67", "TOP2A",
        # MT-high flag
        "MT1E", "MT1X",
    ]
    seen_set = set()
    KNOWN_MARKERS_ORDERED = [g for g in KNOWN_MARKERS_ORDERED
                              if not (g in seen_set or seen_set.add(g))]

    avail       = (set(adata_s.raw.var_names) if adata_s.raw is not None
                   else set(adata_s.var_names))
    known_valid = [g for g in KNOWN_MARKERS_ORDERED if g in avail]
    known_miss  = [g for g in KNOWN_MARKERS_ORDERED if g not in avail]
    if known_miss:
        print(f"  Known markers absent from .raw ({len(known_miss)}): "
              f"{known_miss[:8]}{'...' if len(known_miss) > 8 else ''}")
    print(f"  Known markers available: {len(known_valid)}/{len(KNOWN_MARKERS_ORDERED)}")

    for res in LEIDEN_RESOLUTIONS:
        key     = f"leiden_r{res}"
        n_clust = res_results.get(res, 0)
        if key not in adata_s.obs.columns or n_clust <= 1 or not known_valid:
            continue
        try:
            sc.settings.vector_friendly = True
            fig_w = max(14, len(known_valid) * 0.38)
            fig_h = max(4,  n_clust * 0.6)

            fig_dp = sc.pl.dotplot(
                adata_s, var_names=known_valid, groupby=key,
                use_raw=True, standard_scale="var",
                dot_max=1.0, cmap="Blues",
                show=False, return_fig=True,
                figsize=(fig_w, fig_h),
                title=f"{cell_type}  |  Known Markers  |  Leiden r={res}",
            )
            fig_dp.savefig(ct_mdir / f"dotplot_known_markers_r{res}.pdf",
                           bbox_inches="tight", dpi=DPI)
            plt.close()
            print(f"  Saved: dotplot_known_markers_r{res}.pdf")

            fig_mp = sc.pl.matrixplot(
                adata_s, var_names=known_valid, groupby=key,
                use_raw=True, standard_scale="var",
                cmap="RdBu_r",
                show=False, return_fig=True,
                figsize=(fig_w, fig_h),
                title=f"{cell_type}  |  Known Markers (matrix)  |  Leiden r={res}",
            )
            fig_mp.savefig(ct_mdir / f"matrixplot_known_markers_r{res}.pdf",
                           bbox_inches="tight", dpi=DPI)
            plt.close()
            print(f"  Saved: matrixplot_known_markers_r{res}.pdf")

        except Exception as e:
            print(f"  Known-marker dotplot r={res} failed: {e}")

    # Feature plots: key markers on within-subset UMAP
    FEATURE_PRIORITY = [
        "CD3D", "CD4", "CD8A", "GNLY", "NKG7", "FCGR3A",
        "CCR7", "TCF7", "GZMB", "GZMK", "PRF1",
        "ITGAE", "CD69", "CXCR6", "RGS1",
        "KLRG1", "CX3CR1",
        "PDCD1", "HAVCR2", "FOXP3",
        "CXCL13", "TOX",
        "TRDC", "KLRB1", "ZBTB16", "KIT", "MKI67",
    ]
    feat_valid = [g for g in FEATURE_PRIORITY if g in avail]
    if feat_valid:
        try:
            sc.settings.vector_friendly = True
            ncols_f = 6
            nrows_f = int(np.ceil(len(feat_valid) / ncols_f))
            fig_f, axes_f = plt.subplots(
                nrows_f, ncols_f, figsize=(ncols_f * 3.5, nrows_f * 3.2))
            axes_ff = np.array(axes_f).ravel()
            for idx, gene in enumerate(feat_valid):
                sc.pl.embedding(
                    adata_s, basis="umap", color=gene,
                    ax=axes_ff[idx], show=False,
                    use_raw=(adata_s.raw is not None),
                    cmap="viridis", vmin=0,
                )
                axes_ff[idx].set_title(gene, fontsize=9, fontweight="bold")
            for idx in range(len(feat_valid), len(axes_ff)):
                axes_ff[idx].set_visible(False)
            plt.suptitle(f"{cell_type} -- Key Marker Feature Plots (use_raw=True)",
                         fontsize=12, y=1.01)
            plt.tight_layout()
            fig_f.savefig(ct_mdir / "featureplot_known_markers.pdf",
                          bbox_inches="tight", dpi=DPI)
            plt.close(fig_f)
            print(f"  Saved: featureplot_known_markers.pdf")
        except Exception as e:
            print(f"  Feature plots failed: {e}")

    # Store DEFAULT_ANNOTATION_RESOLUTION clusters back to main adata.obs
    anno_key = f"leiden_r{DEFAULT_ANNOTATION_RESOLUTION}"
    if anno_key in adata_s.obs.columns:
        main_col = f"subcluster_{ct_safe}"
        adata.obs.loc[mask, main_col] = (
            adata_s.obs[anno_key].astype(str)
            .apply(lambda x: f"{ct_safe}_{x}").values
        )
        n_sub = adata_s.obs[anno_key].nunique()
        subcluster_summary[cell_type] = {
            "n_cells"        : n_cells,
            "annotation_key" : main_col,
            "annotation_res" : DEFAULT_ANNOTATION_RESOLUTION,
            "n_subclusters"  : n_sub,
        }
        print(f"  Stored in adata.obs['{main_col}']: {n_sub} subclusters")

    res_results_global[cell_type] = res_results

    del adata_s
    gc.collect()

print("\n" + "="*60)
print("Subclustering complete. Summary:")
for ct, info in subcluster_summary.items():
    print(f"  {ct}: {info['n_subclusters']} subclusters -> obs['{info['annotation_key']}']")
print(f"\nReview files in: {marker_dir}")


Subclustering: NK cells
  Cells: 14,084
  Neighbors on X_scvi (n_neighbors=30)...
computing neighbors
    finished (0:01:11)
computing UMAP
    finished (0:00:25)
running Leiden clustering
    finished (0:00:11)
  Leiden r=1.0: 13 clusters
  Computing markers r=1.0 (13 clusters)...
ranking genes
    finished (0:00:37)
    Saved: markers_leiden_r1.0.csv
  Saved: umap_resolutions_overview.pdf
computing PCA
    with n_comps=50
    finished (0:00:01)
Storing dendrogram info using `.uns['dendrogram_leiden_r1.0']`
  Saved: dotplot_datadriven_r1.0.pdf
  Known markers available: 65/65
  Saved: dotplot_known_markers_r1.0.pdf
  Known-marker dotplot r=1.0 failed: PolyQuadMesh.set() got an unexpected keyword argument 'color_map'
  Saved: featureplot_known_markers.pdf
  Stored in adata.obs['subcluster_nk_cells']: 13 subclusters

Subclustering: CD4 T cells
  Cells: 5,876
  Neighbors on X_scvi (n_neighbors=30)...
computing neighbors
    finished (0:00:00)
computing UMAP
    finished (0:00:24)
runnin

## 3b. Export Dotplot Underlying Data as CSV

For each cell type x resolution, saves three tables:
  - dotplot_data_r*_mean_expr_raw.csv     : log1p mean per cluster (unscaled)
  - dotplot_data_r*_mean_expr_scaled.csv  : 0-1 per-gene scaled (= dotplot color)
  - dotplot_data_r*_pct_expressing.csv    : fraction cells > 0 (= dot size)

Rows = clusters, Columns = known marker genes.
Open in Excel or R alongside PDFs for annotation.

In [ ]:
def _get_expr_matrix(adata_s, genes):
    """Return (dense_matrix, genes_used) from .raw if available."""
    if adata_s.raw is not None:
        src  = adata_s.raw
        idx  = src.var_names.get_indexer(genes)
        ok   = idx >= 0
        genes_used = [genes[i] for i, v in enumerate(ok) if v]
        X    = src.X[:, idx[ok]]
    else:
        idx  = adata_s.var_names.get_indexer(genes)
        ok   = idx >= 0
        genes_used = [genes[i] for i, v in enumerate(ok) if v]
        X    = adata_s.X[:, idx[ok]]
    if sparse.issparse(X):
        X = np.asarray(X.todense())
    return X.astype(np.float32), genes_used

_KNOWN_MARKERS_CSV = [
    "CD3D", "CD3E", "CD4", "CD8A", "CD8B",
    "GNLY", "NKG7", "NCAM1", "FCGR3A",
    "TRDC", "TRGC1", "KLRB1", "ZBTB16", "KIT", "IL7R",
    "CCR7", "SELL", "TCF7", "LEF1", "MAL", "IL6R",
    "GZMK", "GZMA", "GZMH", "GZMB", "PRF1", "IFNG", "TNF",
    "ITGAE", "ITGA1", "CD69", "CXCR6", "RGS1", "ZNF683",
    "KLRG1", "CX3CR1", "S1PR5", "ZEB2", "FGFBP2",
    "PDCD1", "HAVCR2", "TIGIT", "LAG3", "TOX", "ENTPD1",
    "CXCR5", "CXCL13", "TOX2",
    "FOXP3", "IL2RA", "CTLA4",
    "TBX21", "RORC", "IL23R", "CCR6",
    "XCL1", "XCL2", "AREG",
    "ISG15", "IFIT1", "MX1",
    "MKI67", "TOP2A", "MT1E", "MT1X",
]
_seen_csv = set()
_KNOWN_MARKERS_CSV = [g for g in _KNOWN_MARKERS_CSV
                      if not (g in _seen_csv or _seen_csv.add(g))]

print("Exporting dotplot underlying data as CSV...")

for cell_type, info in subcluster_summary.items():
    ct_safe = sanitize_colname(cell_type)
    ct_mdir = marker_dir / ct_safe
    mask    = adata.obs[SPLIT_KEY] == cell_type
    adata_s = adata[mask].copy()

    avail_s     = (set(adata_s.raw.var_names) if adata_s.raw is not None
                   else set(adata_s.var_names))
    known_valid = [g for g in _KNOWN_MARKERS_CSV if g in avail_s]

    if not known_valid:
        print(f"  {cell_type}: no known markers in data -- skipping CSV")
        del adata_s; gc.collect()
        continue

    X_full, genes_used = _get_expr_matrix(adata_s, known_valid)

    for res in LEIDEN_RESOLUTIONS:
        key     = f"leiden_r{res}"
        n_clust = res_results_global.get(cell_type, {}).get(res, 0)
        if key not in adata_s.obs.columns or n_clust <= 1:
            continue

        clusters        = adata_s.obs[key].cat.categories.tolist()
        rows_mean_raw   = {}
        rows_pct        = {}

        for clust in clusters:
            cmask          = (adata_s.obs[key] == clust).values
            X_c            = X_full[cmask, :]
            rows_mean_raw[clust] = X_c.mean(axis=0)
            rows_pct[clust]      = (X_c > 0).mean(axis=0)

        df_mean_raw = pd.DataFrame(rows_mean_raw, index=genes_used).T
        df_pct      = pd.DataFrame(rows_pct,      index=genes_used).T

        gene_min   = df_mean_raw.min(axis=0)
        gene_range = (df_mean_raw.max(axis=0) - gene_min).replace(0, 1)
        df_scaled  = (df_mean_raw - gene_min) / gene_range

        for df, suffix in [
            (df_mean_raw.round(4), "mean_expr_raw"),
            (df_scaled.round(4),   "mean_expr_scaled"),
            (df_pct.round(4),      "pct_expressing"),
        ]:
            df.index.name = "cluster"
            df.to_csv(ct_mdir / f"dotplot_data_r{res}_{suffix}.csv")

        print(f"  {cell_type} r={res}: "
              f"{len(clusters)} clusters x {len(genes_used)} genes -- 3 CSVs saved")

    del adata_s, X_full
    gc.collect()

print(f"\nCSV export complete -> {marker_dir}/<cell_type>/dotplot_data_r*.csv")

## 4. Resolution Override + Print Annotation Templates

[P1-3 fix] ANNOTATION_RES_OVERRIDE is applied HERE, before printing templates.
This ensures cluster IDs in the printed templates always match the resolution
the user will actually annotate. Do not move the override to a later cell.

Steps:
  1. Fill ANNOTATION_RES_OVERRIDE below if needed
  2. Run this cell -- templates are printed with final cluster IDs
  3. Copy the printed templates into Part 5 annotation maps

In [ ]:
# ============================================================================
# Optional: override annotation resolution per cell type
# This is applied BEFORE printing templates so IDs are always consistent.
# Example: {"NK cells": 0.8, "CD8 T cells": 1.0}
# ============================================================================
ANNOTATION_RES_OVERRIDE = {}

# Apply overrides: re-run Leiden at override resolution, update subcluster_summary
for cell_type, override_res in ANNOTATION_RES_OVERRIDE.items():
    if cell_type not in subcluster_summary:
        print(f"[SKIP override] '{cell_type}' not in subcluster_summary")
        continue
    prev_res = subcluster_summary[cell_type]["annotation_res"]
    if override_res == prev_res:
        continue

    print(f"Applying resolution override: {cell_type} -> r={override_res} (was {prev_res})")
    mask    = adata.obs[SPLIT_KEY] == cell_type
    adata_s = adata[mask].copy()
    ct_safe = sanitize_colname(cell_type)

    n_neighbors = min(30, max(5, adata_s.n_obs // 5))
    sc.pp.neighbors(adata_s, use_rep=SUBCLUSTER_REP, n_neighbors=n_neighbors)
    sc.tl.leiden(adata_s, resolution=override_res,
                 key_added=f"leiden_r{override_res}")

    main_col = f"subcluster_{ct_safe}"
    adata.obs.loc[mask, main_col] = (
        adata_s.obs[f"leiden_r{override_res}"].astype(str)
        .apply(lambda x: f"{ct_safe}_{x}").values
    )
    n_sub = adata_s.obs[f"leiden_r{override_res}"].nunique()
    subcluster_summary[cell_type]["annotation_res"] = override_res
    subcluster_summary[cell_type]["annotation_key"] = main_col
    subcluster_summary[cell_type]["n_subclusters"]  = n_sub
    res_results_global[cell_type][override_res]     = n_sub
    print(f"  Updated: {n_sub} subclusters at r={override_res}")

    del adata_s; gc.collect()

# Print annotation templates with FINAL cluster IDs
print("="*70)
print("ANNOTATION TEMPLATES")
print("Copy cluster IDs from the output below into Part 5 annotation maps.")
print("="*70)

for cell_type, info in subcluster_summary.items():
    ct_safe  = sanitize_colname(cell_type)
    anno_key = info["annotation_key"]
    res      = info["annotation_res"]

    print(f"\n# --- {cell_type} (obs['{anno_key}'], r={res}) ---")
    print(f"ANNOTATION_MAP_{ct_safe} = {{")
    if anno_key in adata.obs.columns:
        for c in sorted(adata.obs[anno_key].dropna().unique().tolist()):
            print(f'    "{c}": "",   # fill in label')
    print("}")

print("\n# After filling labels above, paste into Part 5.")

## 5. USER FILLS ANNOTATION MAPS

Instructions:
  1. Review PDFs in subcluster_markers/<cell_type>/
  2. Review CSVs: dotplot_data_r*_{mean_expr_scaled,pct_expressing}.csv
  3. Fill in labels in the dicts below -- use UNLABELED = "Unknown" to
     exclude a cluster from supervision (it will still be kept in the data)
  4. Run this cell, then continue to Part 6

In [ ]:
# ============================================================================
# USER FILLS: replace "" with your annotation label for each cluster
# Cluster IDs follow pattern: {cell_type_safe}_{cluster_number}
# ============================================================================

ANNOTATION_MAP_nk_cells = {
    # "nk_cells_0": "NK Cytotoxic",
    # "nk_cells_1": "NK Resting",
    # "nk_cells_2": "NK IFN-high",
}

ANNOTATION_MAP_cd4_t_cells = {
    # "cd4_t_cells_0": "CD4 Naive/TCM",
    # "cd4_t_cells_1": "CD4 TEM",
    # "cd4_t_cells_2": "CD4 Treg",
}

ANNOTATION_MAP_cd8_t_cells = {
    # "cd8_t_cells_0": "CD8 TRM",
    # "cd8_t_cells_1": "CD8 TEMRA",
    # "cd8_t_cells_2": "CD8 Naive/TCM",
}

# Merge all maps (extend here if you add more cell types)
COMBINED_ANNOTATION_MAP = {}
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_nk_cells)
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_cd4_t_cells)
COMBINED_ANNOTATION_MAP.update(ANNOTATION_MAP_cd8_t_cells)

n_filled = sum(1 for v in COMBINED_ANNOTATION_MAP.values() if v.strip() != "")
n_total  = len(COMBINED_ANNOTATION_MAP)
print(f"Annotation map: {n_filled}/{n_total} entries filled")
if n_filled == 0:
    print("[WARNING] No labels filled. scANVI retrain will fail the >= 2 classes guard.")

## 6. Apply Annotations -> Build NEW_LABELS_KEY

[P0-1 fix] Initialization strategy:
  1. NEW_LABELS_KEY inherits OLD_LABELS_KEY (preserves non-target L2 supervision)
  2. Only cells in TARGET_CELL_TYPES are cleared to Unknown
  3. COMBINED_ANNOTATION_MAP fills back the new labels for target subclusters

Non-target L2 types are never touched and retain their old supervision labels.

In [ ]:
# Backup original labels before overwriting
adata.obs[BACKUP_LABELS_KEY] = adata.obs[OLD_LABELS_KEY].copy()
print(f"Original labels backed up to '{BACKUP_LABELS_KEY}'")

print(f"Building '{NEW_LABELS_KEY}' column...")

# Step 1: Inherit old labels (keeps non-target L2 types supervised)
if OLD_LABELS_KEY in adata.obs.columns:
    adata.obs[NEW_LABELS_KEY] = adata.obs[OLD_LABELS_KEY].astype(str)
    print(f"  Inherited from '{OLD_LABELS_KEY}': "
          f"{adata.obs[OLD_LABELS_KEY].nunique()} old categories")
else:
    adata.obs[NEW_LABELS_KEY] = UNLABELED
    print(f"  '{OLD_LABELS_KEY}' not found -- initializing all cells as '{UNLABELED}'")

# Step 2: Clear only TARGET_CELL_TYPES cells (they will be re-filled from map)
for cell_type in TARGET_CELL_TYPES:
    if cell_type not in subcluster_summary:
        continue
    mask_ct = adata.obs[SPLIT_KEY] == cell_type
    adata.obs.loc[mask_ct, NEW_LABELS_KEY] = UNLABELED

n_cleared = (adata.obs[NEW_LABELS_KEY] == UNLABELED).sum()
print(f"  Cells cleared to '{UNLABELED}' (target types only): {n_cleared:,}")

# Step 3: Fill from COMBINED_ANNOTATION_MAP
n_mapped_labeled   = 0
n_mapped_unlabeled = 0
n_missing_keys     = 0

for cell_type, info in subcluster_summary.items():
    anno_key = info["annotation_key"]
    ct_safe  = sanitize_colname(cell_type)

    if anno_key not in adata.obs.columns:
        print(f"  [WARN] '{anno_key}' not in obs -- {cell_type} stays Unknown")
        continue

    for cluster_id, label in COMBINED_ANNOTATION_MAP.items():
        if not cluster_id.startswith(ct_safe + "_"):
            continue
        label = label.strip() if label.strip() != "" else UNLABELED

        mask = adata.obs[anno_key] == cluster_id
        n    = int(mask.sum())
        if n == 0:
            print(f"  [WARN] Cluster '{cluster_id}' not found in obs['{anno_key}']")
            n_missing_keys += 1
            continue

        adata.obs.loc[mask, NEW_LABELS_KEY] = label
        if label != UNLABELED:
            n_mapped_labeled   += n
        else:
            n_mapped_unlabeled += n

# [P2-1 fix] Accurate per-type counts
n_total_labeled   = (adata.obs[NEW_LABELS_KEY] != UNLABELED).sum()
n_total_unlabeled = (adata.obs[NEW_LABELS_KEY] == UNLABELED).sum()

print(f"\nAssignment summary:")
print(f"  Total cells           : {adata.n_obs:,}")
print(f"  Labeled (non-Unknown) : {n_total_labeled:,}")
print(f"  Unknown               : {n_total_unlabeled:,}")
print(f"  Mapped labeled        : {n_mapped_labeled:,}  (from annotation map, non-Unknown)")
print(f"  Mapped Unknown        : {n_mapped_unlabeled:,} (from annotation map, kept Unknown)")
print(f"  Unresolved map keys   : {n_missing_keys}")
print(f"\nFull label distribution:")
print(adata.obs[NEW_LABELS_KEY].value_counts())

adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].astype("category")
if UNLABELED not in adata.obs[NEW_LABELS_KEY].cat.categories:
    adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].cat.add_categories([UNLABELED])

## 7. Sanity Check UMAP Before Retrain

In [ ]:
# Compute UMAP from X_scanvi if not already present in main adata
if "X_umap" not in adata.obsm:
    print("X_umap not found -- computing from X_scanvi for overview...")
    sc.pp.neighbors(adata, use_rep="X_scanvi", n_neighbors=30)
    sc.tl.umap(adata, min_dist=0.3)

sc.settings.vector_friendly = True
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

sc.pl.embedding(
    adata, basis="umap",
    color=OLD_LABELS_KEY if OLD_LABELS_KEY in adata.obs.columns else SPLIT_KEY,
    title="Before: Old Labels",
    ax=axes[0], show=False, legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(
    adata, basis="umap", color=NEW_LABELS_KEY,
    title=f"After: New Labels ({NEW_LABELS_KEY})",
    ax=axes[1], show=False, legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(
    adata, basis="umap", color=SPLIT_KEY,
    title=f"L2 Cell Type ({SPLIT_KEY})",
    ax=axes[2], show=False, legend_loc="right margin", legend_fontsize=7)

plt.suptitle("Annotation Sanity Check Before scANVI Retrain", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "annotation_sanity_check.pdf", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved: annotation_sanity_check.pdf")
print("\nVerify this plot before continuing to scANVI retrain.")
print("If labels look wrong, edit Part 5 and re-run Parts 5-6.")

## 8. Guard: Validate Label Distribution Before scANVI Retrain

[P0-3 fix] Explicitly block training if labeled classes < 2 or any issue
that would make scANVI supervision meaningless.

In [ ]:
label_counts = adata.obs[NEW_LABELS_KEY].value_counts()
labeled_only = label_counts[label_counts.index != UNLABELED]

print(f"Label validation:")
print(f"  Total classes (incl. Unknown) : {len(label_counts)}")
print(f"  Labeled classes               : {len(labeled_only)}")

if len(labeled_only) == 0:
    raise ValueError(
        f"No labeled classes found in '{NEW_LABELS_KEY}'. "
        f"Fill annotation maps in Part 5 before running scANVI retrain."
    )

if len(labeled_only) < 2:
    raise ValueError(
        f"Only {len(labeled_only)} labeled class in '{NEW_LABELS_KEY}': "
        f"{labeled_only.index.tolist()}. "
        f"scANVI requires at least 2 supervised classes."
    )

smallest_class = labeled_only.idxmin()
min_class_size = int(labeled_only.min())
print(f"  Smallest class: '{smallest_class}' ({min_class_size} cells)")
print(f"\nPer-class distribution:")
print(labeled_only)

# [P0-3 fix] n_samples_per_label: clamp to min_class_size, don't force a floor of 10
if N_SAMPLES_PER_LABEL is None:
    n_samples_per_label = min(100, max(2, int(min_class_size * 0.8)))
    n_samples_per_label = min(n_samples_per_label, min_class_size)
else:
    n_samples_per_label = N_SAMPLES_PER_LABEL

print(f"\nn_samples_per_label (adaptive): {n_samples_per_label}")
print("Label validation passed. Proceeding to scANVI retrain.")

## 9. Reload scVI + Train scANVI with New Labels

In [ ]:
import torch

print(f"Loading scVI model from: {SCVI_MODEL_DIR}")
print(f"  Input adata: {adata.shape} (HVG subset)")

if "counts" not in adata.layers:
    raise ValueError(
        "'counts' layer not found. "
        "Expected HVG-subset h5ad with counts + log1p layers."
    )

if UNLABELED not in adata.obs[NEW_LABELS_KEY].cat.categories:
    adata.obs[NEW_LABELS_KEY] = adata.obs[NEW_LABELS_KEY].cat.add_categories([UNLABELED])

# Reload scVI from saved model directory
vae = scvi.model.SCVI.load(SCVI_MODEL_DIR, adata=adata)
print(f"  scVI loaded: n_latent={vae.module.n_latent}")

# [P0-2 fix] Latent consistency check: compare reloaded scVI against X_scvi
# (not X_scanvi -- different latent spaces, low correlation is expected/normal)
if "X_scvi" not in adata.obsm:
    print("  [INFO] 'X_scvi' not found in obsm -- skipping latent consistency check")
else:
    n_check     = min(500, adata.n_obs)
    test_latent = vae.get_latent_representation(indices=list(range(n_check)))
    existing    = adata.obsm["X_scvi"][:n_check]
    corr        = float(np.corrcoef(test_latent.ravel(), existing.ravel())[0, 1])
    print(f"  X_scvi correlation (reloaded vs stored): {corr:.4f}")
    if corr < 0.95:
        print(f"  [WARN] Low correlation ({corr:.3f}). "
              f"Verify SCVI_MODEL_DIR matches the h5ad training run.")

In [ ]:
print("\nBuilding scANVI from reloaded scVI...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    unlabeled_category = UNLABELED,
    labels_key         = NEW_LABELS_KEY,
)

n_labeled   = int((adata.obs[NEW_LABELS_KEY] != UNLABELED).sum())
n_unlabeled = int((adata.obs[NEW_LABELS_KEY] == UNLABELED).sum())
label_cats  = sorted([c for c in adata.obs[NEW_LABELS_KEY].cat.categories
                       if c != UNLABELED])
print(f"  labels_key       : {NEW_LABELS_KEY}")
print(f"  labeled cells    : {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  unknown cells    : {n_unlabeled:,} ({n_unlabeled/adata.n_obs*100:.1f}%)")
print(f"  supervision classes ({len(label_cats)}): {label_cats}")

In [ ]:
print(f"\nTraining scANVI "
      f"(max_epochs={SCANVI_EPOCHS}, batch_size={BATCH_SIZE}, "
      f"n_samples_per_label={n_samples_per_label})...")
lvae.train(
    max_epochs              = SCANVI_EPOCHS,
    batch_size              = BATCH_SIZE,
    train_size              = 0.9,
    early_stopping          = True,
    early_stopping_patience = 20,
    n_samples_per_label     = n_samples_per_label,
)
print("scANVI training complete")

In [ ]:
adata.obsm["X_scanvi_refined"]       = lvae.get_latent_representation()
adata.obs["scanvi_pred_refined"]      = lvae.predict()
scanvi_proba                          = lvae.predict(soft=True)
adata.obs["scanvi_pred_prob_refined"] = (
    scanvi_proba.max(axis=1).values.astype(np.float32)
)

print(f"X_scanvi_refined: {adata.obsm['X_scanvi_refined'].shape}")
concordance = (
    adata.obs["scanvi_pred_refined"].astype(str) ==
    adata.obs[NEW_LABELS_KEY].astype(str)
).mean()
print(f"Overall concordance (pred vs label): {concordance:.3f}")
print(f"\nPrediction confidence:\n{adata.obs['scanvi_pred_prob_refined'].describe()}")

low_conf = (adata.obs["scanvi_pred_prob_refined"] < 0.5).sum()
print(f"Low-confidence cells (<0.5): {low_conf:,} ({low_conf/adata.n_obs*100:.1f}%)")

print("\nPer-class concordance:")
for cls in label_cats:
    mask = adata.obs[NEW_LABELS_KEY].astype(str) == cls
    if mask.sum() == 0:
        continue
    acc = (adata.obs.loc[mask, "scanvi_pred_refined"].astype(str) == cls).mean()
    print(f"  {cls:<40}: {acc:.3f}  (n={mask.sum():,})")

In [ ]:
# Save training loss curve BEFORE del lvae
try:
    fig_loss, ax_loss = plt.subplots(figsize=(8, 4))
    train_hist = lvae.history.get("train_loss_epoch", None)
    if train_hist is not None:
        ax_loss.plot(train_hist.values.flatten(), label="train ELBO")
    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("ELBO"); ax_loss.legend()
    ax_loss.set_title("scANVI Retrain -- Training Loss")
    fig_loss.savefig(fig_dir / "scanvi_retrain_training_loss.pdf",
                     bbox_inches="tight", dpi=DPI)
    plt.close(fig_loss)
    print("Saved: scanvi_retrain_training_loss.pdf")
except Exception as e:
    print(f"  Training loss plot failed: {e}")

# Save new scANVI model
scanvi_model_dir_new = str(output_dir / "tnk_scanvi_ref_model_retrain")
lvae.save(scanvi_model_dir_new, overwrite=True)
pd.Series(adata.var_names.tolist()).to_csv(
    Path(scanvi_model_dir_new) / "var_names.csv", index=False, header=False)
print(f"scANVI model saved: {scanvi_model_dir_new}/")

del vae, lvae
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory released")

## 10. Post-Retrain Visualization

In [ ]:
print("Computing neighbors + UMAP on X_scanvi_refined...")
sc.pp.neighbors(adata, use_rep="X_scanvi_refined", n_neighbors=30)
sc.tl.umap(adata, min_dist=0.3, spread=1.0, key_added="X_umap_refined")

In [ ]:
sc.settings.vector_friendly = True
fig, axes = plt.subplots(2, 3, figsize=(24, 14))

sc.pl.embedding(adata, basis="umap_refined", color=NEW_LABELS_KEY,
                title=f"New Labels ({NEW_LABELS_KEY})",
                ax=axes[0, 0], show=False,
                legend_loc="right margin", legend_fontsize=6)

sc.pl.embedding(adata, basis="umap_refined", color="scanvi_pred_refined",
                title="scANVI Refined Predictions",
                ax=axes[0, 1], show=False,
                legend_loc="right margin", legend_fontsize=6)

sc.pl.embedding(adata, basis="umap_refined", color="scanvi_pred_prob_refined",
                title="Prediction Confidence",
                ax=axes[0, 2], show=False, cmap="RdYlGn", vmin=0, vmax=1)

sc.pl.embedding(adata, basis="umap_refined", color=SPLIT_KEY,
                title=f"L2 Cell Type ({SPLIT_KEY})",
                ax=axes[1, 0], show=False,
                legend_loc="right margin", legend_fontsize=7)

sc.pl.embedding(adata, basis="umap_refined", color=BATCH_KEY,
                title=f"Batch ({BATCH_KEY})",
                ax=axes[1, 1], show=False,
                legend_loc="right margin", legend_fontsize=5)

concordance_col = (
    adata.obs["scanvi_pred_refined"].astype(str) ==
    adata.obs[NEW_LABELS_KEY].astype(str)
).map({True: "correct", False: "mismatch"}).astype("category")
adata.obs["_pred_concordance"] = concordance_col
sc.pl.embedding(adata, basis="umap_refined", color="_pred_concordance",
                title="Prediction Concordance",
                ax=axes[1, 2], show=False,
                palette={"correct": "#2ecc71", "mismatch": "#e74c3c"})

plt.suptitle("Post-Retrain Overview -- scANVI Refined", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "umap_post_retrain_overview.pdf", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved: umap_post_retrain_overview.pdf")

In [ ]:
MARKER_GENES = [
    "CD3D", "CD4", "CD8A", "GNLY", "NKG7", "FCGR3A",
    "CCR7", "SELL", "TCF7",
    "GZMB", "PRF1", "GZMK",
    "ITGAE", "CD69", "CXCR6", "RGS1",
    "KLRG1", "CX3CR1",
    "PDCD1", "HAVCR2", "FOXP3",
    "CXCL13", "TOX",
    "TRDC", "KLRB1", "ZBTB16", "KIT", "MKI67",
]
avail_raw    = set(adata.raw.var_names) if adata.raw is not None else set(adata.var_names)
marker_valid = [g for g in MARKER_GENES if g in avail_raw]

sc.settings.vector_friendly = True
ncols = 5
nrows = int(np.ceil(len(marker_valid) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes_flat  = np.array(axes).ravel()

for i, gene in enumerate(marker_valid):
    sc.pl.embedding(adata, basis="umap_refined", color=gene,
                    ax=axes_flat[i], show=False,
                    use_raw=(adata.raw is not None),
                    cmap="viridis", vmin=0)
    axes_flat[i].set_title(gene, fontsize=10, fontweight="bold")
for i in range(len(marker_valid), len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.suptitle("T/NK Key Markers -- Refined UMAP (use_raw=True)", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(fig_dir / "featureplot_refined_umap.pdf", bbox_inches="tight", dpi=DPI)
plt.close(fig)
print("Saved: featureplot_refined_umap.pdf")

## 11. uns Metadata + Cleanup + Save

In [ ]:
import anndata as _anndata
_anndata.settings.allow_write_nullable_strings = True

# Remove temp concordance column
if "_pred_concordance" in adata.obs.columns:
    adata.obs.drop(columns=["_pred_concordance"], inplace=True)

adata.uns["retrain_params"] = {
    "script_version"          : "v1.1",
    "base_input_h5ad"         : INPUT_H5AD,
    "scvi_model_dir"          : SCVI_MODEL_DIR,
    "scvi_retrained"          : False,   # [P1-4 fix] scVI NOT retrained
    "scanvi_model_dir_new"    : scanvi_model_dir_new,
    "split_key"               : SPLIT_KEY,
    "target_cell_types"       : TARGET_CELL_TYPES,
    "subcluster_rep"          : SUBCLUSTER_REP,
    "default_anno_res"        : DEFAULT_ANNOTATION_RESOLUTION,
    "annotation_res_override" : ANNOTATION_RES_OVERRIDE,
    "leiden_resolutions"      : LEIDEN_RESOLUTIONS,
    "old_labels_key"          : OLD_LABELS_KEY,
    "new_labels_key"          : NEW_LABELS_KEY,
    "unlabeled_category"      : UNLABELED,
    "scanvi_epochs"           : SCANVI_EPOCHS,
    "n_samples_per_label"     : n_samples_per_label,
    "batch_key"               : BATCH_KEY,
    "scanvi_label_categories" : sorted(
        [c for c in adata.obs[NEW_LABELS_KEY].cat.categories if c != UNLABELED]
    ),
    "subcluster_summary"      : {
        ct: {k: v for k, v in info.items()} for ct, info in subcluster_summary.items()
    },
    "combined_annotation_map" : COMBINED_ANNOTATION_MAP,
}

# Convert label columns to category dtype before write
cat_cols = [NEW_LABELS_KEY, "scanvi_pred_refined", SPLIT_KEY, BATCH_KEY]
if OLD_LABELS_KEY in adata.obs.columns:
    cat_cols.append(OLD_LABELS_KEY)
for col in cat_cols:
    if col in adata.obs.columns:
        adata.obs[col] = adata.obs[col].astype("category")

# Rename any reserved '_index' column
for attr in ("obs", "var"):
    df = getattr(adata, attr)
    if "_index" in df.columns:
        setattr(adata, attr, df.rename(columns={"_index": "orig_index"}))

In [ ]:
out_h5ad = output_dir / "adata_tnk_scanvi_ref_retrain_v1_1.h5ad"
print(f"Writing {out_h5ad.name}...")
print(f"  .X shape  : {adata.shape}")
if adata.raw:
    print(f"  .raw shape: {adata.raw.n_obs} x {adata.raw.n_vars}")
print(f"  .layers   : {list(adata.layers.keys())}")
print(f"  .obsm     : {list(adata.obsm.keys())}")
adata.write_h5ad(out_h5ad, compression="gzip", compression_opts=9)
print(f"Saved: {out_h5ad}")

## 12. Final Summary

In [ ]:
print("="*70)
print("PIPELINE COMPLETE (v1.1)")
print("="*70)

print(f"\nOutput h5ad       : {out_h5ad}")
print(f"scVI model        : {SCVI_MODEL_DIR}/  (UNCHANGED -- original reference)")
print(f"scANVI model      : {scanvi_model_dir_new}/  (new refined labels)")
print(f"Subcluster markers: {marker_dir}/")
print(f"Figures           : {fig_dir}/")

print(f"\nNew annotation column : '{NEW_LABELS_KEY}'")
print(f"Prediction column     : 'scanvi_pred_refined'")
print(f"Latent space          : 'X_scanvi_refined'")
print(f"UMAP                  : 'umap_refined'")

print(f"\nLabel distribution:")
print(adata.obs[NEW_LABELS_KEY].value_counts())

print(f"\nscArches query checklist (for future query scripts):")
print(f"  1. labels_key           = '{NEW_LABELS_KEY}'")
print(f"  2. scvi_model_dir       = '{SCVI_MODEL_DIR}'")
print(f"  3. scanvi_model_dir     = '{scanvi_model_dir_new}'")
print(f"  4. Expected label space = {sorted([c for c in adata.obs[NEW_LABELS_KEY].cat.categories if c != UNLABELED])}")
print(f"  5. Unlabeled category   = '{UNLABELED}'")